<a href="https://colab.research.google.com/github/TalCordova/RS_Coller_TAU_26B/blob/master/notebook3_collaborative_filtering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook 3: Collaborative Filtering — Item-Item & User-User

In notebooks 1 and 2 we prepared our data and built a ranking pipeline. But we never asked: where does the signal come from? How does the system know that two items are related, or that two users have similar taste?

This notebook answers that question. Collaborative filtering is the idea that user behavior itself is the signal — if many users who liked A also liked B, that tells us something. No item features, no user profiles. Just patterns in interactions.

We will build two classic CF methods:
- **Item-item** — *“Because you watched X...”* — find items similar to ones the user already liked
- **User-user** — *“Users like you also watched...”* — find users with similar taste and recommend what they liked

---
> **How to use this notebook:** Read the markdown, run each cell in order, and do the interactive prompts as you go:
> - 🔮 **Predict** — guess the answer *before* running the next cell
> - 🔧 **Tweak** — change a value, re-run, and see what moves
> - 🔍 **Interpret** — read an output and answer the question
> - 🏋️ **Your turn** — write the code yourself
>
> Stuck on a 🏋️ cell? Each one is followed by a **🔑 Solution** cell — in Colab it shows only a title bar, so **double-click it to reveal** the code. Text answers hide behind a *“🔑 Reveal answer”* toggle.

## Section 0: Setup

Run the cell below to install and import everything we need.

In [ ]:
!pip install numpy pandas scikit-learn matplotlib seaborn --quiet

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.neighbors import NearestNeighbors

# Nicer plots
plt.rcParams['figure.figsize'] = (10, 4)
sns.set_style('whitegrid')
print("✅ All packages loaded")

---
## Section 1: Load the Data

We use the **MovieLens Small** dataset (~100k ratings from 610 users on 9,742 movies).  
It contains two files:
- `ratings.csv` — `userId`, `movieId`, `rating`, `timestamp`
- `movies.csv` — `movieId`, `title`, `genres`



In [ ]:
import urllib.request, zipfile, io, os

# Download and unzip MovieLens small dataset
url = "https://files.grouplens.org/datasets/movielens/ml-latest-small.zip"
print("Downloading MovieLens dataset...")
with urllib.request.urlopen(url) as r:
    zf = zipfile.ZipFile(io.BytesIO(r.read()))
    zf.extractall(".")
print("✅ Done")

In [ ]:
ratings = pd.read_csv("ml-latest-small/ratings.csv")
movies  = pd.read_csv("ml-latest-small/movies.csv")

print(f"Ratings: {ratings.shape[0]:,} rows | Movies: {movies.shape[0]:,} rows")
ratings.head()

In [ ]:
movies.head()

In [ ]:
# ── Train / test split (user-temporal) ────────────────────────────────────
# We build all similarities on the TRAIN set only, and keep TEST aside to
# sanity-check our rating predictions later (same split idea as Notebook 1).
def user_temporal_split(df, test_frac=0.2):
    """For each user, hold out their most recent interactions as test."""
    train_list, test_list = [], []
    for _, user_df in df.groupby('userId'):
        user_df = user_df.sort_values('timestamp')
        n_test  = max(1, int(len(user_df) * test_frac))
        test_list.append(user_df.iloc[-n_test:])
        train_list.append(user_df.iloc[:-n_test])
    return pd.concat(train_list), pd.concat(test_list)

train, test = user_temporal_split(ratings)
print(f"Train: {len(train):,} ratings  |  Test: {len(test):,} ratings")

In [ ]:
# Build the utility matrix (TRAIN only — never peek at test when building similarities)
utility_matrix = train.pivot_table(index='userId', columns='movieId', values='rating')

n_users, n_items = utility_matrix.shape
n_ratings = train.shape[0]
sparsity = 1 - n_ratings / (n_users * n_items)

print(f"Matrix shape  : {n_users} users × {n_items} items")
print(f"Total cells   : {n_users * n_items:,}")
print(f"Filled cells  : {n_ratings:,}")
print(f"Sparsity      : {sparsity:.1%}")  # expect ~98%

---
## Section 2: Item-Item Collaborative Filtering

**The core idea:** Two movies are similar if *the same users rated them similarly*.  

We represent each movie as a vector of user ratings (a column in the utility matrix), then measure similarity between those vectors using **cosine similarity**.

$$\text{cos}(A, B) = \frac{A \cdot B}{\|A\| \|B\|}$$

A value of 1 means identical direction (very similar), 0 means orthogonal (unrelated), -1 means opposite.

In [ ]:
# Fill NaN with 0 (unrated = neutral, after mean-centering this is reasonable)
X = utility_matrix.fillna(0).values          # shape: (n_users, n_movies)
movie_ids_index = list(utility_matrix.columns)
user_ids_index  = list(utility_matrix.index)

print(f"Rating matrix X shape: {X.shape}")
print(f"  {X.shape[0]} users  ×  {X.shape[1]} movies")

In [ ]:
# Fit a kNN model on the TRANSPOSED matrix — each item is a row
# (sklearn's NearestNeighbors finds the k closest rows)
model_item = NearestNeighbors(metric='cosine', algorithm='brute', n_neighbors=11)
model_item.fit(X.T)   # X.T shape: (n_movies, n_users)
print("Item-item kNN model fitted ✅")

In [ ]:
def find_similar_movies(movie_id, k=10):
    """Return the k most similar movies to movie_id."""
    if movie_id not in movie_ids_index:
        print(f"Movie {movie_id} not found in training data.")
        return []

    idx = movie_ids_index.index(movie_id)
    distances, indices = model_item.kneighbors(
        X.T[idx].reshape(1, -1), n_neighbors=k + 1
    )
    similar_indices    = indices.flatten()[1:]    # skip index 0 = the movie itself
    similarity_scores  = 1 - distances.flatten()[1:]

    results = pd.DataFrame({
        'movieId'   : [movie_ids_index[i] for i in similar_indices],
        'similarity': similarity_scores
    })
    results = results.merge(movies[['movieId', 'title', 'genres']], on='movieId', how='left')
    return results

### 🔮 Predict — what's most similar to Toy Story?

We're about to ask item-item CF for the 10 movies most similar to **Toy Story (1995)**. Before you run it:

Will the top neighbours be other **animated kids' movies** (Toy Story 2, Lion King…), or something else?

<details>
<summary>🔑 Reveal answer</summary>

A bit of both — Toy Story 2 shows up, but so do **Jurassic Park, Independence Day, Star Wars**. The signal here is *“which movies did the same users rate?”*, and almost everyone in the dataset rated the big 90s blockbusters. We'll see why that matters next.
</details>

In [ ]:
# ── Try it out ────────────────────────────────────────────────────────────
# Movie ID 1 = Toy Story (1995)
toy_story_id = 1
similar = find_similar_movies(toy_story_id, k=10)

anchor_title = movies.loc[movies['movieId'] == toy_story_id, 'title'].values[0]
print(f"Because you watched: {anchor_title}\n")
print(similar[['title', 'genres', 'similarity']].to_string(index=False))

### 🔍 Interpret — why do blockbusters keep showing up?

Several of Toy Story's “most similar” movies are just very popular films from the same era, not films that are genuinely *about* the same thing.

Why does raw co-rating similarity drift toward popular items? (Hang onto your answer — we return to this as **popularity bias** in Section 4.)

<details>
<summary>🔑 Reveal answer</summary>

Popular movies were rated by almost everyone, so their rating vectors overlap with *everything*. Cosine similarity on raw co-ratings therefore rewards popularity, not topical relevance. This is **popularity bias** baked right into the similarity signal — niche items rarely surface.
</details>

In [ ]:
# ── Try another movie ─────────────────────────────────────────────────────
# Movie ID 296 = Pulp Fiction (1994)
pulp_fiction_id = 296
similar_pf = find_similar_movies(pulp_fiction_id, k=10)

anchor_title_pf = movies.loc[movies['movieId'] == pulp_fiction_id, 'title'].values[0]
print(f"Because you watched: {anchor_title_pf}\n")
print(similar_pf[['title', 'genres', 'similarity']].to_string(index=False))

### 🔧 Tweak — how many neighbours?

`find_similar_movies` defaults to `k=10`. Try calling it with a couple of different `k` values on a movie of your choice:

```python
find_similar_movies(296, k=3)    # Pulp Fiction — just the 3 nearest
find_similar_movies(296, k=50)   # cast a much wider net
```

How does the neighbour list change as `k` grows? What happens to the *relevance* of the furthest neighbours?

### 🏋️ Exercise: Predict a Rating (Item-Item)

Given a user and a movie, we can **predict the rating** the user would give to that movie using the weighted average of their ratings on similar movies:

$$\hat{r}_{u,i} = \frac{\sum_{j \in N(i)} \text{sim}(i,j) \cdot r_{u,j}}{\sum_{j \in N(i)} |\text{sim}(i,j)|}$$

Where:
- $N(i)$ = the k most similar items to item $i$
- $r_{u,j}$ = user $u$'s actual rating on item $j$ (only items the user has rated)
- $\text{sim}(i,j)$ = cosine similarity between items $i$ and $j$

**Complete the function below:**

In [ ]:
def predict_rating_item_item(user_id, movie_id, k=10):
    """Predict the rating user_id would give movie_id using item-item CF."""
    if user_id not in user_ids_index:
        return None

    # the user's actual ratings, and their mean (the fallback)
    user_ratings = utility_matrix.loc[user_id].dropna()
    user_mean    = user_ratings.mean()

    # 1. the k movies most similar to movie_id
    neighbours = find_similar_movies(movie_id, k=k)
    if len(neighbours) == 0:
        return user_mean

    # 2. keep only neighbours the user has actually rated
    rated = neighbours[neighbours['movieId'].isin(user_ratings.index)]
    if len(rated) == 0:
        return user_mean

    # 3. similarity-weighted average of those ratings
    sims  = rated['similarity'].values
    rates = user_ratings.loc[rated['movieId']].values
    denom = abs(sims).sum()
    if denom == 0:
        return user_mean
    return float((sims * rates).sum() / denom)

In [ ]:
# ── Test your function ────────────────────────────────────────────────────
# Pick a (user, movie) pair that exists in the TEST set
sample = test[['userId', 'movieId', 'rating']].iloc[0]
uid, mid, actual = int(sample['userId']), int(sample['movieId']), sample['rating']

predicted = predict_rating_item_item(uid, mid, k=10)

movie_title = movies.loc[movies['movieId'] == mid, 'title'].values
movie_title = movie_title[0] if len(movie_title) else f"Movie {mid}"

print(f"User        : {uid}")
print(f"Movie       : {movie_title}")
print(f"Actual      : {actual}")
print(f"Predicted   : {predicted:.2f}" if predicted is not None else "Predicted   : None (function not yet implemented)")

---
## Section 3: User-User Collaborative Filtering

**The core idea:** Two users are similar if they rated the *same movies* in a *similar way*.  

The algorithm is symmetric to item-item — instead of comparing movie vectors, we compare user vectors (rows in the utility matrix).

> **Item-item vs User-user — when does each work better?**  
> - **User-user** tends to be more personalised but scales poorly (comparing every pair of users is expensive as the user base grows)  
> - **Item-item** is faster to update (item catalogs change less often than user behavior) and was famously used by Amazon

Both are forms of **memory-based CF** — they don't learn model parameters, they just look up the database at prediction time.

In [ ]:
# Fit a kNN model on the ORIGINAL matrix — each user is a row
model_user = NearestNeighbors(metric='cosine', algorithm='brute', n_neighbors=11)
model_user.fit(X)    # X shape: (n_users, n_movies)
print("User-user kNN model fitted ✅")

In [ ]:
def find_similar_users(user_id, k=10):
    """Return the k most similar users to user_id."""
    if user_id not in user_ids_index:
        print(f"User {user_id} not found in training data.")
        return []

    idx = user_ids_index.index(user_id)
    distances, indices = model_user.kneighbors(
        X[idx].reshape(1, -1), n_neighbors=k + 1
    )
    similar_indices   = indices.flatten()[1:]
    similarity_scores = 1 - distances.flatten()[1:]

    return pd.DataFrame({
        'userId'    : [user_ids_index[i] for i in similar_indices],
        'similarity': similarity_scores
    })

In [ ]:
# ── Try it out ────────────────────────────────────────────────────────────
example_user_id = user_ids_index[0]
similar_users = find_similar_users(example_user_id, k=10)

print(f"Users most similar to User {example_user_id}:\n")
print(similar_users.to_string(index=False))

### 🏋️ Exercise: Predict a Rating (User-User)

The prediction formula for user-user CF is:

$$\hat{r}_{u,i} = \bar{r}_u + \frac{\sum_{v \in N(u)} \text{sim}(u,v) \cdot (r_{v,i} - \bar{r}_v)}{\sum_{v \in N(u)} |\text{sim}(u,v)|}$$

Where:
- $N(u)$ = the k users most similar to user $u$
- $r_{v,i}$ = neighbor $v$'s rating on item $i$
- $\bar{r}_u$, $\bar{r}_v$ = mean ratings of users $u$ and $v$ (we add/subtract these because we mean-centered the data)

> Notice the difference from item-item: here we **add back the user's mean** $\bar{r}_u$ so the prediction is in the original rating scale.

**Complete the function below:**

In [ ]:
def predict_rating_user_user(user_id, movie_id, k=10):
    """Predict the rating user_id would give movie_id using user-user CF."""
    if user_id not in user_ids_index:
        return None

    user_ratings = utility_matrix.loc[user_id].dropna()
    user_mean    = user_ratings.mean()

    neighbours = find_similar_users(user_id, k=k)
    if len(neighbours) == 0:
        return user_mean

    num, den = 0.0, 0.0
    for _, row in neighbours.iterrows():
        v, sim    = int(row['userId']), row['similarity']
        v_ratings = utility_matrix.loc[v].dropna()
        if movie_id in v_ratings.index:
            num += sim * (v_ratings[movie_id] - v_ratings.mean())
            den += abs(sim)

    if den == 0:
        return user_mean

    # add the user's own mean back to the weighted neighbour deviations
    prediction = user_mean + num / den
    return float(prediction)

In [ ]:
# ── Test your function ────────────────────────────────────────────────────
predicted_uu = predict_rating_user_user(uid, mid, k=10)

print(f"User        : {uid}")
print(f"Movie       : {movie_title}")
print(f"Actual      : {actual}")
print(f"Item-item   : {predict_rating_item_item(uid, mid):.2f}" if predict_rating_item_item(uid, mid) else "Item-item   : not implemented")
print(f"User-user   : {predicted_uu:.2f}" if predicted_uu else "User-user   : not implemented")

---
## Summary

In this notebook, we introduced two types of CF based recommender systems.

| | Item-Item | User-User |
|---|---|---|
| **Similarity** | Between items (columns) | Between users (rows) |
| **Prediction** | Weighted avg of ratings on similar items | Weighted avg of similar users' ratings (mean-adjusted) |
| **Use case** | "Because you watched X" | "Users like you also watched X" |

Both approaches are **memory-based** — no model parameters are learned, predictions come directly from the rating data.

In practice, systems handle this with fallbacks:
- **New user** — recommend globally popular items until enough interactions accumulate. If demographic information is available (age, location, preferences from onboarding), popular items can be filtered or ranked within a matching demographic group.
- **New item** — if item properties are available (genre, description, tags), use content-based similarity to find items with similar properties and surface the new item to users who liked those. Without any properties, the item simply cannot be recommended until someone interacts with it.

## Section 4: More Things to Consider

These are the real-world challenges every RS practitioner faces. No code needed — but understanding these is as important as knowing the algorithms.

### 4.1 Cold Start

**The problem:** CF and BPR both rely on interaction history. A brand new user (or a brand new item) has no interactions — there's nothing to learn from.

This is called the **cold start problem**.

**What can we do?**

| Approach | How it works |
|---|---|
| **Content-based fallback** | Use item features (genre, description) instead of interactions. |
| **Popularity-based** | Recommend the most popular items globally. Not personalized, but better than nothing. |
| **Onboarding** | Ask new users to rate a few items up front (Netflix does this). Gathers minimal signal before CF kicks in. |
| **Hybrid** | Combine CF scores with content-based scores using a weighted blend. |

> **Note:** The filtering we applied in Notebook 1 (removing users with < 5 interactions) actually *hides* the cold start problem from our training data. In production, cold start users exist — your model will encounter them.

### 4.2 Popularity Bias

**The problem:** The long tail of items (niche movies, obscure books) almost never gets recommended.

Why? Because CF and BPR learn from interactions — popular items have *many more* interactions, so the model learns them much better. The system becomes self-reinforcing: popular items get recommended → they get more interactions → they get recommended even more.

The filtering step (users with ≥ 5 interactions) makes this worse: it preferentially keeps users who interact with popular items.

### 🔍 Interpret — connecting back to Toy Story

Remember the blockbusters that crashed Toy Story's neighbour list? The plot below makes that bias quantitative: it compares the popularity of **recommended** items against the **whole catalog**.

Predict the shape first: will the orange (recommended) distribution sit to the *left* (more niche) or the *right* (more popular) of the blue (all items)?

<details>
<summary>🔑 Reveal answer</summary>

The recommended items skew **right** — toward the popular head of the catalog. The long tail of niche movies is almost never recommended, which is exactly the popularity-bias feedback loop described above.
</details>

In [ ]:
# ── Visualize popularity bias in recommendations ──────────────────────────
# How popular are the items we recommend vs. the full catalog?
item_popularity = ratings.groupby('movieId')['rating'].count()

def recommend_for_user(user_id, k=10):
    """A cheap item-item recommendation: movies similar to the user's top-rated movie."""
    user_ratings = utility_matrix.loc[user_id].dropna()
    if len(user_ratings) == 0:
        return []
    fav_movie = user_ratings.idxmax()              # the user's highest-rated movie
    recs = find_similar_movies(fav_movie, k=k)
    return list(recs['movieId']) if len(recs) else []

# Collect recommendations for a sample of users
rec_movie_ids = []
for uid in user_ids_index[:50]:
    rec_movie_ids.extend(recommend_for_user(uid, k=10))

rec_popularity = item_popularity.reindex(rec_movie_ids).dropna()
all_popularity = item_popularity

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].hist(np.log1p(all_popularity.values), bins=50, color='steelblue',
             edgecolor='white', alpha=0.7, label='All items')
axes[0].hist(np.log1p(rec_popularity.values), bins=50, color='coral',
             edgecolor='white', alpha=0.7, label='Recommended items')
axes[0].set_xlabel("log(number of ratings)")
axes[0].set_ylabel("Count")
axes[0].set_title("Popularity: All Items vs. Recommended Items")
axes[0].legend()

# Cumulative: what fraction of ratings go to the top-X% of items?
pop_sorted  = item_popularity.sort_values(ascending=False).reset_index(drop=True)
cum_items   = np.arange(1, len(pop_sorted)+1) / len(pop_sorted) * 100
cum_ratings = pop_sorted.cumsum() / pop_sorted.sum() * 100

axes[1].plot(cum_items, cum_ratings, color='steelblue', linewidth=2)
axes[1].plot([0, 100], [0, 100], 'k--', linewidth=1, label='Perfect equality')
idx20 = np.searchsorted(cum_items/100, 0.2)
axes[1].axvline(20, color='red', linestyle='--', alpha=0.7)
axes[1].axhline(cum_ratings.iloc[idx20], color='red', linestyle='--', alpha=0.7,
                label=f"Top 20% of items → {cum_ratings.iloc[idx20]:.0f}% of ratings")
axes[1].set_xlabel("Cumulative % of items")
axes[1].set_ylabel("Cumulative % of ratings")
axes[1].set_title("Long Tail — Rating Concentration")
axes[1].legend()

plt.tight_layout()
plt.show()

**What can we do about popularity bias?**

- **Re-ranking with diversity penalty:** After generating a Top-K list, apply a post-processing step that penalizes popular items and boosts niche ones.
- **Inverse popularity sampling:** When sampling negatives for BPR training, over-sample popular items as negatives — this teaches the model that "popular ≠ always relevant for this user."
- **Exposure-aware evaluation:** Beyond HR and NDCG, track what fraction of your catalog actually gets recommended.

### 4.3 Beyond Accuracy

HR@K and NDCG@K measure whether we recommended something the user *would have* interacted with. But a good recommender system needs more than accuracy.

| Property | Definition | Example failure |
|---|---|---|
| **Diversity** | Are the recommendations varied? | Recommending 10 action movies to an action fan |
| **Serendipity** | Does the system surprise the user with something they wouldn't have found themselves? | Only recommending obvious sequels |
| **Fairness** | Are all users and all items treated equitably? | Female directors never appearing in top-10 |
| **Coverage** | What fraction of the item catalog ever gets recommended? | 95% of all recommendations are the same 100 movies |

> A system that only recommends what you already know you like is not very useful. The best RS finds the intersection of *what the user would enjoy* and *what they haven't yet discovered*.